In [3]:
from dotenv import load_dotenv
import os

from langchain_community.graphs import Neo4jGraph

In [ ]:
load_dotenv('.env', override= True)

NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

OPENAI_ENDPOINT = os.getenv('OPENAI_BASE_URL') + '/embeddings'

In [4]:
kg = Neo4jGraph(
    url= NEO4J_URI,
    username= NEO4J_USERNAME,
    password= NEO4J_PASSWORD,
    database= NEO4J_DATABASE
)

C:\Users\hp\AppData\Local\Temp\ipykernel_6464\1279993764.py:1: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jGraph``.
  kg = Neo4jGraph(


In [ ]:
# Création d'un vecteur index nommé 'movie_tagline_embedding" s'il n'existe pas
# l'index sera appliqué uniquement sur le noeud ayant le label'Movie' 
# et plus précisément sur  leur propriété tagembedding
# avec les options suivants: 
#   * taille du vecteur embediing = 1536 et
#   * la fonction de similarité utilisée est cosine


kg.query("""
  CREATE VECTOR INDEX movie_tagline_embeddings IF NOT EXISTS
  FOR (m:Movie) ON (m.taglineEmbedding) 
  OPTIONS { indexConfig: {
    `vector.dimensions`: 1536,
    `vector.similarity_function`: 'cosine'
  }}"""
)



[]

In [7]:
kg.query("""
        SHOW VECTOR INDEXES
         """)

[{'id': 4,
  'name': 'movie_tagline_embeddings',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'VECTOR',
  'entityType': 'NODE',
  'labelsOrTypes': ['Movie'],
  'properties': ['taglineEmbedding'],
  'indexProvider': 'vector-2026.06',
  'owningConstraint': None,
  'lastRead': None,
  'readCount': 0}]

In [10]:
# VECTORISATION
# Cela permet de convertir les noms des films (texte) en vecteur de nombres (embeddings)
# La taille du vecteur sera 1536

# Par exemple le film dont tagline= "resident evil" sera transformée en:
# taglineembedding = [0.25434, -0.25353465, .....] ou len(taglineebedding) = 1536

kg.query("""
    MATCH (movie:Movie) WHERE movie.tagline IS NOT NULL
    WITH movie, genai.vector.encode(
        movie.tagline, 
        "OpenAI", 
        {
          token: $openAiApiKey,
          endpoint: $openAiEndpoint
        }) AS vector
    CALL db.create.setNodeVectorProperty(movie, "taglineEmbedding", vector)
    """, 
    params={"openAiApiKey":OPENAI_API_KEY, "openAiEndpoint": OPENAI_ENDPOINT} )

NameError: name 'OPENAI_ENDPOINT' is not defined

In [ ]:
result = kg.query("""
    MATCH (m:Movie) 
    WHERE m.tagline IS NOT NULL
    RETURN m.tagline, m.taglineEmbedding
    LIMIT 1
    """
)

[{'id': 4,
  'name': 'movie_tagline_embeddings',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'VECTOR',
  'entityType': 'NODE',
  'labelsOrTypes': ['Movie'],
  'properties': ['taglineEmbedding'],
  'indexProvider': 'vector-2026.06',
  'owningConstraint': None,
  'lastRead': None,
  'readCount': 0}]

In [ ]:
result[0]['m.tagline']   # --> 'Welcome to the Real World'

In [ ]:
result[0]['m.taglineEmbedding'][:10] 



"""[0.017445066943764687,
-0.005481892731040716,
-0.002013522433117032,
-0.025571243837475777,
-0.014404304325580597,
0.016737302765250206,
-0.017078077420592308,
0.000485358847072348,
-0.025217361748218536,
-0.029516370967030525]"""

In [ ]:
len(result[0]['m.taglineEmbedding']) # --> 1536

In [ ]:
# autre exemple de film
# 10 films, on récupére le deuxième film

result = kg.query("""
    MATCH (m:Movie) 
    WHERE m.tagline IS NOT NULL
    RETURN m.tagline, m.taglineEmbedding
    LIMIT 10
    """
)
print(result[1]['m.tagline'])
print(result[1]['m.taglineEmbedding'][:10])
print("\n", len(result[1]['m.taglineEmbedding']))

In [ ]:
# Effectuer SIMILARITY SEARCH
question = "What movies are about love?"

In [ ]:
kg.query("""
    WITH genai.vector.encode(
        $question, 
        "OpenAI", 
        {
          token: $openAiApiKey,
          endpoint: $openAiEndpoint
        }) AS question_embedding
    CALL db.index.vector.queryNodes(
        'movie_tagline_embeddings', 
        $top_k, 
        question_embedding
        ) YIELD node AS movie, score
    RETURN movie.title, movie.tagline, score
    """, 
    params={"openAiApiKey":OPENAI_API_KEY,
            "openAiEndpoint": OPENAI_ENDPOINT,
            "question": question,
            "top_k": 5
            })

<pre>[{'movie.title': 'Joe Versus the Volcano',
  'movie.tagline': 'A story of love, lava and burning desire.',
  'score': 0.9062913656234741},
 {'movie.title': 'As Good as It Gets',
  'movie.tagline': 'A comedy from the heart that goes for the throat.',
  'score': 0.9022631645202637},
 {'movie.title': 'Snow Falling on Cedars',
  'movie.tagline': 'First loves last. Forever.',
  'score': 0.9013131856918335},
 {'movie.title': 'Sleepless in Seattle',
  'movie.tagline': 'What if someone you never met, someone you never saw, someone you never knew was the only someone for you?',
  'score': 0.8945093154907227},
 {'movie.title': 'When Harry Met Sally',
  'movie.tagline': 'Can two friends sleep together and still love each other in the morning?',
  'score': 0.8942364454269409}]